In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms
from torch.optim import Adam
from tqdm import tqdm


In [2]:
class UNet(nn.Module):
  def __init__(self, in_channels=3, out_channels=1):
    super().__init__()

    # Encoder
    self.enc1 = ConvBlock(in_channels, 64)
    self.enc2 = ConvBlock(64, 128)
    self.enc3 = ConvBlock(128, 256)
    self.enc4 = ConvBlock(256, 512)
    self.enc5 = ConvBlock(512, 1024)

    # Decoder
    self.dec5 = ConvTransposeBlock(1024, 512)
    self.dec4 = ConvTransposeBlock(512, 256)
    self.dec3 = ConvTransposeBlock(256, 128)
    self.dec2 = ConvTransposeBlock(128, 64)
    self.dec1 = ConvTransposeBlock(64, out_channels)

    # Self-Attention
    self.mhsa1 = MHSAHead(64)
    self.mhsa2 = MHSAHead(128)
    self.mhsa3 = MHSAHead(256)
    self.mhsa4 = MHSAHead(512)
    self.mhsa5 = MHSAHead(1024)

  def forward(self, x):
    # Encoder
    x1 = self.enc1(x)
    x2 = self.enc2(x1)
    x3 = self.enc3(x2)
    x4 = self.enc4(x3)
    x5 = self.enc5(x4)

    # Self-Attention
    x1_att = self.mhsa1(x1)
    x2_att = self.mhsa2(x2)
    x3_att = self.mhsa3(x3)
    x4_att = self.mhsa4(x4)
    x5_att = self.mhsa5(x5)

    # Decoder
    x = self.dec5(x5_att)
    x = torch.cat([x, x4_att], dim=1)
    x = self.dec4(x)
    x = torch.cat([x, x3_att], dim=1)
    x = self.dec3(x)
    x = torch.cat([x, x2_att], dim=1)
    x = self.dec2(x)
    x = torch.cat([x, x1_att], dim=1)
    x = self.dec1(x)

    return x


In [3]:
class ConvBlock(nn.Module):
  def __init__(self, in_channels, out_channels):
    super().__init__()

    self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
    self.bn1 = nn.BatchNorm2d(out_channels)
    self.relu = nn.ReLU()
    self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
    self.bn2 = nn.BatchNorm2d(out_channels)

  def forward(self, x):
    x = self.conv1(x)
    x = self.bn1(x)
    x = self.relu(x)
    x = self.conv2(x)
    x = self.bn2(x)


In [4]:
class ConvTransposeBlock(nn.Module):
  def __init__(self, in_channels, out_channels):
    super().__init__()

    self.conv = nn.ConvTranspose2d(in_channels, out_channels, kernel_size=3, padding=1)
    self.bn = nn.BatchNorm2d(out_channels)
    self.relu = nn.ReLU()

  def forward(self, x):
    x = self.conv(x)
    x = self.bn(x)
    x = self.relu(x)
    return x


In [5]:
class MHSAHead(nn.Module):
  def __init__(self, in_channels, num_heads=8, d_model=None):
    super().__init__()
    if d_model is None:
      d_model = in_channels

    self.num_heads = num_heads
    self.d_model = d_model

    # Linear transformations for queries, keys, and values
    self.q_linear = nn.Linear(in_channels, d_model * num_heads)
    self.k_linear = nn.Linear(in_channels, d_model * num_heads)
    self.v_linear = nn.Linear(in_channels, d_model * num_heads)

    # Weight matrix for the final projection
    self.out_proj = nn.Linear(d_model * num_heads, d_model)

  def forward(self, x):
    # Project input features to queries, keys, and values
    q = self.q_linear(x).view(x.size(0), -1, self.num_heads, self.d_model)
    k = self.k_linear(x).view(x.size(0), -1, self.num_heads, self.d_model)
    v = self.v_linear(x).view(x.size(0), -1, self.num_heads, self.d_model)

    # Transpose for efficient attention calculation
    q = q.transpose(2, 0, 1)  # (num_heads, batch_size, seq_len, d_model)
    k = k.transpose(2, 0, 1)  # (num_heads, batch_size, seq_len, d_model)
    v = v.transpose(2, 0, 1)  # (num_heads, batch_size, seq_len, d_model)

    # Scaled Dot-Product Attention
    scores = torch.bmm(q, k.transpose(-2, -1)) / math.sqrt(self.d_model)  # (num_heads, batch_size, seq_len, seq_len)
    scores = F.softmax(scores, dim=-1)  # Apply softmax for attention weights

    # Context vector calculation
    context = torch.bmm(scores, v)  # (num_heads, batch_size, seq_len, d_model)

    # Concatenate heads and apply final projection
    context = context.transpose(2, 0, 1).contiguous().view(x.size(0), -1, self.d_model * self.num_heads)
    output = self.out_proj(context)

    return output


In [7]:
import torch
import torch.nn as nn
import torch.optim as Adam
import torchvision
from torch.utils.data import DataLoader
from tqdm import tqdm

# تنظیمات دستگاه
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# تنظیمات هایپرپارامتر
batch_size = 16
learning_rate = 0.001
num_epochs = 100

# مسیر مجموعه داده
#data_dir = "data/MRI"

# بارگیری مجموعه داده
#train_dataset = MRIDataset(data_dir, split="train")
#train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# تعریف مدل
model = UNet().to(device)

# تعریف تابع Loss
#criterion = IoULoss().to(device)

# تعریف Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)


In [8]:
from torchsummary import summary
summary(model, (3, 224, 224))

AttributeError: 'NoneType' object has no attribute 'size'

In [ ]:
for epoch in range(num_epochs):
  print(f"Epoch {epoch + 1}/{num_epochs}")

  running_loss = 0.0
  for i, (images, masks) in enumerate(tqdm(train_loader)):
    images = images.to(device)
    masks = masks.to(device)

    # پیش بینی مدل
    outputs = model(images)

    # محاسبه Loss
    loss = criterion(outputs, masks)
    running_loss += loss.item()

    # به روز رسانی پارامترهای مدل
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

  # محاسبه Loss میانگین در هر دوره
  epoch_loss = running_loss / len(train_loader)
  print(f"Loss: {epoch_loss:.4f}")

# ذخیره مدل آموزش دیده
torch.save(model.state_dict(), "model.pth")


In [ ]:
#inspect.getsource(MRIDataset)